In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv("../data/processed/customers_clean.csv")

print("Dataset shape:", df.shape)
print(df.head())

Dataset shape: (7043, 21)
   customerID  gender  SeniorCitizen Partner Dependents  tenure PhoneService  \
0  7590-VHVEG  Female              0     Yes         No       1           No   
1  5575-GNVDE    Male              0      No         No      34          Yes   
2  3668-QPYBK    Male              0      No         No       2          Yes   
3  7795-CFOCW    Male              0      No         No      45           No   
4  9237-HQITU  Female              0      No         No       2          Yes   

      MultipleLines InternetService OnlineSecurity  ... DeviceProtection  \
0  No phone service             DSL             No  ...               No   
1                No             DSL            Yes  ...              Yes   
2                No             DSL            Yes  ...               No   
3  No phone service             DSL            Yes  ...              Yes   
4                No     Fiber optic             No  ...               No   

  TechSupport StreamingTV StreamingM

In [2]:
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier

# Remove rows without a target
df = df.dropna(subset=["Churn"]).copy()

# Convert the target to 0 and 1
y = (
    df["Churn"]
    .astype(str)
    .str.strip()
    .str.lower()
    .map({"yes": 1, "no": 0})
)

if y.isna().any():
    raise ValueError("Unexpected values in the Churn column.")

# Exclude the target and customer identifier
X = df.drop(columns=["Churn", "customerID"], errors="ignore")

# Identify feature types
numeric_features = X.select_dtypes(
    include=["number"]
).columns.tolist()

categorical_features = X.select_dtypes(
    include=["object", "category", "bool"]
).columns.tolist()

# Preprocessing for numerical features
numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

# Preprocessing for categorical features
categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("numeric", numeric_pipeline, numeric_features),
    ("categorical", categorical_pipeline, categorical_features)
])

# Split before fitting preprocessing
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

# Combine preprocessing and model
churn_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", RandomForestClassifier(
        n_estimators=200,
        max_depth=12,
        min_samples_leaf=5,
        class_weight="balanced",
        random_state=42,
        n_jobs=-1
    ))
])

# Train the complete pipeline
churn_pipeline.fit(X_train, y_train)

print("Complete pipeline trained successfully!")

C:\Users\Admin\AppData\Local\Temp\ipykernel_16460\3194069881.py:31: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_features = X.select_dtypes(


Complete pipeline trained successfully!


In [3]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix
)

y_pred = churn_pipeline.predict(X_test)
y_prob = churn_pipeline.predict_proba(X_test)[:, 1]

print("Accuracy:", round(accuracy_score(y_test, y_pred), 3))
print("Precision:", round(
    precision_score(y_test, y_pred, zero_division=0), 3
))
print("Recall:", round(
    recall_score(y_test, y_pred, zero_division=0), 3
))
print("F1-score:", round(
    f1_score(y_test, y_pred, zero_division=0), 3
))
print("ROC-AUC:", round(roc_auc_score(y_test, y_prob), 3))

print("\nConfusion matrix:")
print(confusion_matrix(y_test, y_pred))

Accuracy: 0.759
Precision: 0.531
Recall: 0.775
F1-score: 0.63
ROC-AUC: 0.841

Confusion matrix:
[[779 256]
 [ 84 290]]


In [4]:
import joblib
import os

# Create a folder for saved models
os.makedirs("../models", exist_ok=True)

# Save preprocessing and model together
joblib.dump(
    churn_pipeline,
    "../models/churn_pipeline.joblib"
)

print("Model saved successfully!")

Model saved successfully!


In [5]:
loaded_pipeline = joblib.load(
    "../models/churn_pipeline.joblib"
)

# Predict using the saved pipeline
loaded_predictions = loaded_pipeline.predict(X_test)

# Predict churn probabilities
loaded_probabilities = loaded_pipeline.predict_proba(X_test)[:, 1]

print("First 10 predictions:", loaded_predictions[:10])
print("First 10 churn probabilities:",
      loaded_probabilities[:10].round(3))

# Confirm that loading preserved the predictions
assert np.array_equal(y_pred, loaded_predictions)

print("Verification successful: predictions match!")

First 10 predictions: [0 1 0 1 0 1 1 0 0 1]
First 10 churn probabilities: [0.028 0.84  0.121 0.504 0.072 0.78  0.582 0.206 0.012 0.609]
Verification successful: predictions match!


In [6]:
# Take one real customer row from the test set
sample_customer = X_test.iloc[[0]]

prediction = loaded_pipeline.predict(sample_customer)[0]
probability = loaded_pipeline.predict_proba(sample_customer)[0, 1]

print("Predicted outcome:",
      "Likely to churn" if prediction == 1 else "Likely to stay")

print(f"Estimated churn probability: {probability:.2%}")

Predicted outcome: Likely to stay
Estimated churn probability: 2.76%
